# 07 - Comparacion y GradCAM

Comparo los cuatro modelos entrenados y uso GradCAM para ver que regiones de la hoja esta mirando el mejor modelo cuando clasifica. Es una forma de verificar que el modelo esta aprendiendo lo correcto y no haciendo trampa con el fondo o la iluminacion.

In [ ]:
!pip install tensorflow-datasets gdown scikit-learn seaborn --quiet

In [ ]:
import os

WORK_PATH = '/content/plantvillage'
os.makedirs(WORK_PATH, exist_ok=True)

In [ ]:
import gdown
gdown.download_folder(
    'https://drive.google.com/drive/folders/1OCOyDSR9C3TCzLthsMxJCmy6wcjM3pxs',
    output=WORK_PATH,
    quiet=True
)

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import json
from sklearn.metrics import classification_report, confusion_matrix

print('tensorflow:', tf.__version__)
print('gpu disponible:', tf.config.list_physical_devices('GPU'))

tf.random.set_seed(42)
np.random.seed(42)

## Carga de resultados

Cargo los JSON que guardo cada notebook con su accuracy en test y armo la tabla comparativa.

In [ ]:
with open(f'{WORK_PATH}/dataset_info.json') as f:
    ds_info = json.load(f)

NUM_CLASSES = ds_info['num_classes']
CLASS_NAMES = ds_info['class_names']

# nombres de archivo tal como los guardaron los notebooks 03-06
archivos_resultados = [
    ('resultados_baseline.json',    'CNN baseline'),
    ('resultados_vgg16.json',       'VGG16'),
    ('resultados_resnet50.json',    'ResNet50'),
    ('resultados_inceptionv3.json', 'InceptionV3'),
]

resultados = []
for archivo, nombre in archivos_resultados:
    ruta = f'{WORK_PATH}/{archivo}'
    if os.path.exists(ruta):
        with open(ruta) as f:
            r = json.load(f)
        resultados.append(r)
    else:
        print(f'no encontrado: {ruta}')

print(f'{len(resultados)} resultados cargados')

## Tabla comparativa

In [ ]:
print(f'{"modelo":<25} {"test accuracy":>14}')
print('-' * 41)
for r in resultados:
    acc = r['test_acc']
    print(f'{r["modelo"]:<25} {acc:>14.4f}  ({acc*100:.2f}%)')

mejor = max(resultados, key=lambda x: x['test_acc'])
print(f'\nmejor modelo: {mejor["modelo"]} con {mejor["test_acc"]*100:.2f}%')

## Grafica comparativa de accuracy

In [ ]:
nombres = [r['modelo'] for r in resultados]
accs    = [r['test_acc'] * 100 for r in resultados]
colores = ['#888888', '#2E75B6', '#70AD47', '#ED7D31']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(nombres, accs, color=colores[:len(nombres)], width=0.5, edgecolor='white')
ax.set_ylim(0, 105)
ax.set_ylabel('accuracy en test (%)')
ax.set_title('comparativa de modelos - PlantVillage')
for bar, acc in zip(bars, accs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{acc:.2f}%',
        ha='center', va='bottom', fontsize=10
    )
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/comparativa_modelos.png', dpi=150)
plt.show()

## GradCAM

Cargo el mejor modelo de transfer learning y construyo un modelo auxiliar para calcular los gradientes de la clase predicha respecto a los mapas de la ultima capa convolucional. Eso me dice que zonas de la imagen activaron la decision.

In [ ]:
# selecciono el mejor modelo de transfer learning segun los resultados
MODELOS_TL = [
    ('vgg16_best.keras',       'vgg16',       'block5_conv3',    224,
     tf.keras.applications.vgg16.preprocess_input),
    ('resnet50_best.keras',    'resnet50',     'conv5_block3_out', 224,
     tf.keras.applications.resnet50.preprocess_input),
    ('inceptionv3_best.keras', 'inceptionv3',  'mixed10',         299,
     tf.keras.applications.inception_v3.preprocess_input),
]

# selecciono el mejor modelo de transfer learning segun los resultados cargados
tl_results = [r for r in resultados if r['modelo'] != 'CNN baseline']
mejor_tl = max(tl_results, key=lambda x: x['test_acc']) if tl_results else None

if mejor_tl:
    print(f'mejor modelo TL: {mejor_tl["modelo"]} ({mejor_tl["test_acc"]*100:.2f}%)')
else:
    print('no hay resultados de transfer learning disponibles')

In [ ]:
config_por_modelo = {
    'VGG16':       ('vgg16_best.keras',       'vgg16',      'block5_conv3',     224,
                    tf.keras.applications.vgg16.preprocess_input),
    'ResNet50':    ('resnet50_best.keras',     'resnet50',   'conv5_block3_out', 224,
                    tf.keras.applications.resnet50.preprocess_input),
    'InceptionV3': ('inceptionv3_best.keras',  'inceptionv3', 'mixed10',         160,
                    tf.keras.applications.inception_v3.preprocess_input),
}

archivo_modelo, nombre_base, nombre_capa_conv, img_size_gradcam, preprocess_fn = \
    config_por_modelo[mejor_tl['modelo']]

best_model = tf.keras.models.load_model(
    f'{WORK_PATH}/{archivo_modelo}',
    custom_objects={'preprocess_input': preprocess_fn}
)

print(f'modelo cargado: {archivo_modelo}')
print(f'capa convolucional para GradCAM: {nombre_capa_conv}')

## Implementacion de GradCAM

El modelo auxiliar expone la salida de la ultima capa convolucional. Con GradientTape calculo los gradientes de la clase predicha respecto a esos mapas; el promedio por canal me da la importancia de cada filtro.

In [ ]:
def make_gradcam_heatmap(img_array, model, base_layer_name, conv_layer_name, preprocess_fn):
    base_submodel = model.get_layer(base_layer_name)
    conv_layer    = base_submodel.get_layer(conv_layer_name)

    # modelo auxiliar desde la entrada de la base hasta [salida conv, salida base]
    # ambos tensores estan en el mismo grafo interno, por eso esto funciona
    feat_model = tf.keras.Model(
        inputs=base_submodel.inputs,
        outputs=[conv_layer.output, base_submodel.output]
    )

    # preprocesar manualmente: equivale a la capa Lambda del modelo
    # las capas de augmentation son no-op en inferencia
    img_preprocessed = preprocess_fn(tf.cast(img_array, tf.float32))

    with tf.GradientTape() as tape:
        conv_outputs, base_features = feat_model(img_preprocessed, training=False)
        tape.watch(conv_outputs)
        # aplicar la cabeza del modelo externo (GAP -> Dense -> Dropout -> Dense)
        x = base_features
        for layer in model.layers:
            if layer.__class__.__name__ in ('GlobalAveragePooling2D', 'Dense', 'Dropout'):
                x = layer(x, training=False)
        predictions = x
        pred_index  = tf.argmax(predictions[0])
        class_score = predictions[:, pred_index]

    grads        = tape.gradient(class_score, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index), predictions.numpy()[0]


def overlay_gradcam(img_rgb_01, heatmap, alpha=0.4):
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_color = cm.jet(heatmap_uint8)[:, :, :3]
    superimposed  = heatmap_color * alpha + img_rgb_01 * (1 - alpha)
    return np.clip(superimposed, 0, 1)


print('funciones GradCAM definidas')

## Visualizacion de GradCAM

Muestro la imagen original y el heatmap superpuesto. Verde = prediccion correcta, rojo = error.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

(_, _, ds_test_raw), _ = tfds.load(
    'plant_village',
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    with_info=True,
    as_supervised=True,
    shuffle_files=True,
    read_config=tfds.ReadConfig(shuffle_seed=42)
)

# tomo 9 imagenes del test set
muestras = list(ds_test_raw.take(9))

fig, ejes = plt.subplots(9, 2, figsize=(8, 36))

for i, (img_raw, label_raw) in enumerate(muestras):
    true_label = int(label_raw.numpy())

    # preparo la imagen al tamano del modelo
    img_f32    = tf.cast(img_raw, tf.float32)
    img_resz   = tf.image.resize(img_f32, [img_size_gradcam, img_size_gradcam])
    img_01     = img_resz.numpy() / 255.0          # para visualizar
    img_batch  = tf.expand_dims(img_resz, 0)       # en [0,255], el modelo preprocesa internamente

    heatmap, pred_idx, _ = make_gradcam_heatmap(
        img_batch, best_model, nombre_base, nombre_capa_conv, preprocess_fn
    )

    # redimensiono el heatmap al tamano de la imagen
    heatmap_full = tf.image.resize(
        tf.expand_dims(tf.expand_dims(heatmap, -1), 0),
        [img_size_gradcam, img_size_gradcam]
    )[0, :, :, 0].numpy()

    overlay = overlay_gradcam(img_01, heatmap_full)

    color_titulo = 'green' if pred_idx == true_label else 'red'

    ejes[i][0].imshow(img_01)
    ejes[i][0].set_title(f'real: {CLASS_NAMES[true_label][:25]}', fontsize=7)
    ejes[i][0].axis('off')

    ejes[i][1].imshow(overlay)
    ejes[i][1].set_title(f'pred: {CLASS_NAMES[pred_idx][:25]}', fontsize=7, color=color_titulo)
    ejes[i][1].axis('off')

plt.suptitle(f'GradCAM - {mejor_tl["modelo"]}', fontsize=12)
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/gradcam_ejemplos.png', dpi=150)
plt.show()

## Analisis de errores

Busco donde se equivoca el modelo y aplico GradCAM sobre esos casos para ver si hay un patron en los errores.

In [ ]:
def preprocess_for_eval(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, [img_size_gradcam, img_size_gradcam])
    return image, label

ds_test_eval = (
    ds_test_raw
    .map(preprocess_for_eval, num_parallel_calls=AUTOTUNE)
    .batch(32)
    .prefetch(AUTOTUNE)
)

# recorro el dataset una vez para que pred y labels queden alineados
y_pred_list, y_true_list = [], []
for images, labels in ds_test_eval:
    preds = best_model(images, training=False)
    y_pred_list.extend(np.argmax(preds.numpy(), axis=1))
    y_true_list.extend(labels.numpy())

y_pred = np.array(y_pred_list)
y_true = np.array(y_true_list)

test_acc_final = np.mean(y_pred == y_true)
print(f'accuracy en test ({mejor_tl["modelo"]}): {test_acc_final:.4f} ({test_acc_final*100:.2f}%)')

In [ ]:
# las 10 clases con mas errores
errores_por_clase = {}
for vt, vp in zip(y_true, y_pred):
    if vt != vp:
        errores_por_clase[CLASS_NAMES[vt]] = errores_por_clase.get(CLASS_NAMES[vt], 0) + 1

top_errores = sorted(errores_por_clase.items(), key=lambda x: x[1], reverse=True)[:10]

clases_err = [x[0] for x in top_errores]
conteos_err = [x[1] for x in top_errores]

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(clases_err[::-1], conteos_err[::-1], color='#E05252')
ax.set_xlabel('numero de errores')
ax.set_title(f'clases con mas errores - {mejor_tl["modelo"]}')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/errores_por_clase.png', dpi=150)
plt.show()

In [ ]:
# aplico GradCAM sobre los primeros errores del test set
indices_error = np.where(y_pred != y_true)[0]
print(f'total de errores: {len(indices_error)} de {len(y_true)} ({len(indices_error)/len(y_true)*100:.1f}%)')

# tomo las primeras 6 imagenes con error para visualizar con GradCAM
errores_a_mostrar = 6
muestras_error = []
contador = 0
for img_raw, label_raw in ds_test_raw:
    if contador in indices_error[:errores_a_mostrar]:
        muestras_error.append((img_raw.numpy(), int(label_raw.numpy()), y_pred[contador]))
    contador += 1
    if len(muestras_error) >= errores_a_mostrar:
        break

fig, ejes = plt.subplots(errores_a_mostrar, 2, figsize=(8, errores_a_mostrar * 4))

for i, (img_np, true_lbl, pred_lbl) in enumerate(muestras_error):
    img_f32   = tf.cast(img_np, tf.float32)
    img_resz  = tf.image.resize(img_f32, [img_size_gradcam, img_size_gradcam])
    img_01    = img_resz.numpy() / 255.0
    img_batch = tf.expand_dims(img_resz, 0)

    heatmap, _, _ = make_gradcam_heatmap(
        img_batch, best_model, nombre_base, nombre_capa_conv, preprocess_fn
    )
    heatmap_full = tf.image.resize(
        tf.expand_dims(tf.expand_dims(heatmap, -1), 0),
        [img_size_gradcam, img_size_gradcam]
    )[0, :, :, 0].numpy()

    overlay = overlay_gradcam(img_01, heatmap_full)

    ejes[i][0].imshow(img_01)
    ejes[i][0].set_title(f'real: {CLASS_NAMES[true_lbl][:28]}', fontsize=7)
    ejes[i][0].axis('off')

    ejes[i][1].imshow(overlay)
    ejes[i][1].set_title(f'pred: {CLASS_NAMES[pred_lbl][:28]}', fontsize=7, color='red')
    ejes[i][1].axis('off')

plt.suptitle(f'GradCAM sobre errores - {mejor_tl["modelo"]}', fontsize=12)
plt.tight_layout()
plt.savefig(f'{WORK_PATH}/gradcam_errores.png', dpi=150)
plt.show()

## Resumen final

El transfer learning supera al baseline porque los pesos de ImageNet ya tienen aprendidas texturas y bordes utiles para hojas. El fine-tuning ajusta esas representaciones al dominio de PlantVillage. GradCAM confirma que el modelo se concentra en las manchas y decoloraciones de la hoja, que es exactamente lo que uno miraria para diagnosticar una enfermedad.

In [ ]:
print('resumen final')
print('=' * 50)
print(f'{"modelo":<25} {"test accuracy":>14}')
print('-' * 41)
for r in sorted(resultados, key=lambda x: x['test_acc'], reverse=True):
    marca = ' <-- mejor' if r['modelo'] == mejor['modelo'] else ''
    print(f'{r["modelo"]:<25} {r["test_acc"]*100:>13.2f}%{marca}')
print()
print('archivos generados:')
for archivo in ['comparativa_modelos.png', 'gradcam_ejemplos.png',
                'gradcam_errores.png', 'errores_por_clase.png']:
    ruta = f'{WORK_PATH}/{archivo}'
    print(f'  {archivo}: {"ok" if os.path.exists(ruta) else "no generado"}')